# Ch.6 — Hyperparameter Tuning for Regression

**Goal**: Systematically optimize all regression hyperparameters to push MAE from $38k → $32k.

| What | Value |
|------|-------|
| Dataset | California Housing (20,640 districts, 8 features) |
| Ch.4 best | Ridge (degree=2, α=1.0) → $38k MAE |
| This chapter target | ~$32k MAE via systematic tuning |

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
)
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import loguniform, uniform

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

In [ ]:
# ── Load and split ─────────────────────────────────────────────────────────
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

## Step 1: Grid Search — Ridge (degree × α)

Joint tuning of polynomial degree and regularization strength.

In [ ]:
# ── Grid search: Ridge (degree × α) ──────────────────────────────────────
ridge_pipe = Pipeline([
    ('poly', PolynomialFeatures(include_bias=False)),
    ('scaler', StandardScaler()),
    ('model', Ridge())
])

ridge_params = {
    'poly__degree': [1, 2, 3],
    'model__alpha': np.logspace(-3, 3, 7)  # 0.001 to 1000
}

ridge_cv = GridSearchCV(
    ridge_pipe, ridge_params, cv=5,
    scoring='neg_mean_absolute_error', n_jobs=-1,
    return_train_score=True
)
ridge_cv.fit(X_train, y_train)

ridge_mae = mean_absolute_error(y_test, ridge_cv.predict(X_test)) * 100_000
print(f"Best Ridge:")
print(f"  degree = {ridge_cv.best_params_['poly__degree']}")
print(f"  α      = {ridge_cv.best_params_['model__alpha']:.4f}")
print(f"  CV MAE = ${-ridge_cv.best_score_ * 100_000:,.0f}")
print(f"  Test MAE = ${ridge_mae:,.0f}")

In [ ]:
# ── Heatmap: degree × α ──────────────────────────────────────────────────
results = pd.DataFrame(ridge_cv.cv_results_)
heatmap_data = results.pivot_table(
    index='param_poly__degree',
    columns='param_model__alpha',
    values='mean_test_score'
)
heatmap_data = -heatmap_data * 100_000

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(heatmap_data, annot=True, fmt=',.0f', cmap='RdYlGn_r',
            ax=ax, cbar_kws={'label': 'CV MAE ($)'})
ax.set_xlabel('α (regularization strength)', fontsize=12)
ax.set_ylabel('Polynomial degree', fontsize=12)
ax.set_title('Ridge: Joint Tuning of Degree × α\n'
             'Green = lower MAE (better)', fontsize=14)
plt.tight_layout()
plt.savefig('img/ridge_grid_search.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 2: Random Search — Elastic Net

60 random trials of α × l1_ratio (Bergstra & Bengio 2012).

In [ ]:
# ── Random search: Elastic Net ────────────────────────────────────────────
en_pipe = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('model', ElasticNet(max_iter=10000))
])

en_params = {
    'model__alpha': loguniform(1e-4, 1),
    'model__l1_ratio': uniform(0, 1)
}

en_cv = RandomizedSearchCV(
    en_pipe, en_params, n_iter=60, cv=5,
    scoring='neg_mean_absolute_error', n_jobs=-1, random_state=42
)
en_cv.fit(X_train, y_train)

en_mae = mean_absolute_error(y_test, en_cv.predict(X_test)) * 100_000
print(f"Best Elastic Net:")
print(f"  α        = {en_cv.best_params_['model__alpha']:.6f}")
print(f"  l1_ratio = {en_cv.best_params_['model__l1_ratio']:.3f}")
print(f"  CV MAE   = ${-en_cv.best_score_ * 100_000:,.0f}")
print(f"  Test MAE = ${en_mae:,.0f}")

In [ ]:
# ── Random search trial visualization ─────────────────────────────────────
en_results = pd.DataFrame(en_cv.cv_results_)

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(
    en_results['param_model__alpha'],
    en_results['param_model__l1_ratio'],
    c=-en_results['mean_test_score'] * 100_000,
    cmap='RdYlGn_r', s=80, alpha=0.8, edgecolors='gray'
)
# Mark best
ax.scatter(
    en_cv.best_params_['model__alpha'],
    en_cv.best_params_['model__l1_ratio'],
    s=200, c='none', edgecolors='red', linewidths=3, marker='*',
    zorder=10, label='Best'
)
ax.set_xscale('log')
ax.set_xlabel('α (log scale)', fontsize=12)
ax.set_ylabel('l1_ratio (0=Ridge, 1=Lasso)', fontsize=12)
ax.set_title('Random Search: 60 Elastic Net Trials\n'
             'Each dot = one trial, color = CV MAE', fontsize=14)
plt.colorbar(scatter, label='CV MAE ($)', ax=ax)
ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig('img/random_search_trials.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 3: Lasso Feature Selection + Ridge Refit

Use Lasso to identify important features, then refit Ridge on just those.

In [ ]:
# ── Lasso feature selection ───────────────────────────────────────────────
lasso_pipe = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('model', Lasso(max_iter=10000))
])

lasso_params = {'model__alpha': np.logspace(-4, -1, 20)}
lasso_cv = GridSearchCV(lasso_pipe, lasso_params, cv=5,
                        scoring='neg_mean_absolute_error', n_jobs=-1)
lasso_cv.fit(X_train, y_train)

# Get selected features
best_lasso = lasso_cv.best_estimator_
lasso_coefs = best_lasso.named_steps['model'].coef_
n_selected = np.sum(lasso_coefs != 0)
n_dropped = np.sum(lasso_coefs == 0)
lasso_mae = mean_absolute_error(y_test, lasso_cv.predict(X_test)) * 100_000

print(f"Best Lasso: α={lasso_cv.best_params_['model__alpha']:.6f}")
print(f"  Features: {n_selected} kept, {n_dropped} dropped")
print(f"  Test MAE: ${lasso_mae:,.0f}")

## Step 4: XGBoost (if installed)

Non-linear regression with tree-based ensemble — no polynomial features needed.

In [ ]:
# ── XGBoost tuning ───────────────────────────────────────────────────────
try:
    from xgboost import XGBRegressor
    from scipy.stats import randint
    
    xgb = XGBRegressor(
        objective='reg:squarederror',
        n_jobs=-1, random_state=42, verbosity=0
    )
    
    xgb_params = {
        'n_estimators': randint(100, 800),
        'max_depth': randint(3, 10),
        'learning_rate': loguniform(0.01, 0.3),
        'subsample': uniform(0.7, 0.3),
        'colsample_bytree': uniform(0.7, 0.3),
        'reg_alpha': uniform(0, 1),
        'reg_lambda': uniform(0, 1),
    }
    
    xgb_cv = RandomizedSearchCV(
        xgb, xgb_params, n_iter=50, cv=5,
        scoring='neg_mean_absolute_error', n_jobs=-1, random_state=42
    )
    xgb_cv.fit(X_train, y_train)
    xgb_mae = mean_absolute_error(y_test, xgb_cv.predict(X_test)) * 100_000
    
    print(f"Best XGBoost:")
    for k, v in xgb_cv.best_params_.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
    print(f"  Test MAE: ${xgb_mae:,.0f}")
    has_xgb = True
except ImportError:
    print("XGBoost not installed. Run: pip install xgboost")
    print("Skipping XGBoost tuning.")
    xgb_mae = None
    has_xgb = False

## Final Model Comparison

All tuned models side by side.

In [ ]:
# ── Final comparison ──────────────────────────────────────────────────────
print("═" * 65)
print("  FINAL MODEL COMPARISON — ALL TUNED")
print("═" * 65)
print(f"  {'Model':<30} {'Test MAE':>12} {'vs $40k':>10}")
print(f"  {'-'*30} {'-'*12} {'-'*10}")

# Ch.4 baseline (untuned)
baseline_pipe = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0))
])
baseline_pipe.fit(X_train, y_train)
baseline_mae = mean_absolute_error(y_test, baseline_pipe.predict(X_test)) * 100_000

models = [
    ('Ch.4 Ridge (α=1, untuned)', baseline_mae),
    ('Ridge (tuned)', ridge_mae),
    ('Elastic Net (tuned)', en_mae),
    ('Lasso (tuned)', lasso_mae),
]
if has_xgb:
    models.append(('XGBoost (tuned)', xgb_mae))

for name, mae_val in models:
    marker = ' ✅' if mae_val < 40_000 else ''
    print(f"  {name:<30} {'${:,.0f}'.format(mae_val):>12} {'${:+,.0f}'.format(mae_val - 40_000):>10}{marker}")

print(f"  {'-'*30} {'-'*12} {'-'*10}")
print(f"  {'🎯 Target':<30} {'<$40,000':>12}")
print("═" * 65)

best_name, best_mae = min(models, key=lambda x: x[1])
improvement = baseline_mae - best_mae
print(f"\n🏆 Best model: {best_name}")
print(f"   Improvement over Ch.4 baseline: ${improvement:,.0f}")

In [ ]:
# ── Full journey bar chart ────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression

# Recompute all chapter baselines
lr1 = LinearRegression().fit(X_train[['MedInc']], y_train)
mae_ch1 = mean_absolute_error(y_test, lr1.predict(X_test[['MedInc']])) * 100_000

sc2 = StandardScaler()
lr2 = LinearRegression().fit(sc2.fit_transform(X_train), y_train)
mae_ch2 = mean_absolute_error(y_test, lr2.predict(sc2.transform(X_test))) * 100_000

poly3 = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])
poly3.fit(X_train, y_train)
mae_ch3 = mean_absolute_error(y_test, poly3.predict(X_test)) * 100_000

chapters = ['Ch.1\nLinear', 'Ch.2\nMultiple', 'Ch.3\nPolynomial',
            'Ch.4\nRidge', 'Ch.6\nTuned']
maes = [mae_ch1, mae_ch2, mae_ch3, baseline_mae, best_mae]

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#f8d7da' if m > 40_000 else '#d4edda' for m in maes]
bars = ax.bar(chapters, maes, color=colors, edgecolor='gray', width=0.6)
ax.axhline(y=40_000, color='green', linestyle='--', linewidth=2, label='$40k target')

for bar, mae_val in zip(bars, maes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f'${mae_val:,.0f}', ha='center', fontsize=11, fontweight='bold')

ax.set_ylabel('MAE ($)', fontsize=12)
ax.set_title('SmartVal AI: Full Regression Journey\n'
             'Ch.1 → Ch.6', fontsize=14)
ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig('img/full_journey.png', dpi=150, bbox_inches='tight')
plt.show()

## Exercises

1. **Optuna Bayesian search**: Install Optuna and implement the Bayesian optimization example from the README. Compare trials-to-best vs RandomizedSearchCV.
2. **Degree 3 + strong Ridge**: Tune α for degree=3 polynomial (164 features). Can regularization tame it better than degree=2?
3. **Feature importance from XGBoost**: Plot `feature_importances_` from the tuned XGBoost model. Which raw features matter most without polynomial expansion?

In [ ]:
# ── Exercise 1 scaffold: Optuna ───────────────────────────────────────────
# TODO: pip install optuna, implement objective function, compare to random search
pass

In [ ]:
# ── Exercise 2 scaffold: degree 3 + Ridge ────────────────────────────────
# TODO: GridSearch α for degree=3, compare best MAE to degree=2
pass

In [ ]:
# ── Exercise 3 scaffold: XGBoost feature importance ──────────────────────
# TODO: Plot xgb.feature_importances_, compare to Lasso's feature selection
pass